# Spark Fundamentals & Data Cleaning — Week 5

Data Engineering 003 — Spark Questions (Q1–Q15)

This notebook covers Spark fundamentals, DataFrame data cleaning, transformations, aggregations, and a final processing pipeline.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

spark = SparkSession.builder.appName("SparkBasics").getOrCreate()
spark

## Q1: Key limitations of MapReduce that make Spark preferred

- MapReduce writes intermediate results to **disk** after every map/reduce stage, causing high I/O overhead.
- No efficient support for **iterative algorithms** (ML, graph processing) since each iteration re-reads/re-writes to disk.
- **Verbose, low-level API** (lots of boilerplate Java code).
- No built-in support for **interactive/ad-hoc queries**.
- Spark instead uses **in-memory computation** (RDDs/DataFrames), a richer high-level API, and a **DAG execution engine** that optimizes the whole pipeline instead of stage-by-stage.

## Q2: In-Memory Computing for iterative ML algorithms

In MapReduce, each iteration of an algorithm (e.g., gradient descent) reads input from disk, processes it, and writes output back to disk — repeated every iteration.

Spark keeps the dataset cached in RAM (via `.cache()` / `.persist()`) across iterations, so subsequent passes reuse the in-memory data instead of re-reading from disk. This drastically cuts I/O latency, making iterative ML training (e.g., MLlib algorithms) much faster.

In [ ]:
# Example: caching a DataFrame for iterative use
df = spark.read.csv("dataset.csv", header=True, inferSchema=True)
df.cache()
df.count()  # triggers caching

## Q3: Remove duplicate rows based on user_id and transaction_date

In [ ]:
df_clean = df.dropDuplicates(["user_id", "transaction_date"])
df_clean.show()

## Q4: Filter region == 'West', group by product_category, average sale_amount

In [10]:
result = (df
          .filter(df.region == "West")
          .groupBy("product_category")
          .avg("sale_amount"))
result.show()

+----------------+------------------+
|product_category|  avg(sale_amount)|
+----------------+------------------+
|  Home & Kitchen| 258.7352941176471|
|          Sports|249.48529411764707|
|       Groceries|          218.4105|
|     Electronics| 233.1935294117647|
|        Clothing| 242.2308695652174|
|           Books|218.42849999999999|
+----------------+------------------+



## Q5: .na.drop() vs .na.fill()

- `.na.drop()` removes rows containing null values (entire row dropped).
- `.na.fill()` replaces null values with a specified value instead of removing rows.

In [11]:
# Fill nulls in 'status' column with 'Unknown'
df_clean = df.na.fill({"status": "Unknown"})
df_clean.show()

+-------+----------------+------+----------------+-----------+---+------------+-----------+-------------------+--------+------+--------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|age|subscription|       city|              email|username| price|store_id|  status|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---+------------+-----------+-------------------+--------+------+--------+--------+-------------------+
|  U1330|      10/11/2024|  East|        Clothing|     199.46| 24|     Premium| Georgetown|user330@example.com|user_330|689.66|    S014|Inactive|2024-10-20 04:00:00|
|  U1369|      03/08/2024| North|       Groceries|       NULL| 23|        Free|   Franklin|user369@example.com|user_369|376.07|    S006| Pending|2025-02-04 22:00:00|
|  U1144|      2024-06-29|  East|           Books|       96.1| 50|        Free|Springfield|user144@example.com|user_144|819.82|    S002|  Active|2024-07-30 04:00:00|
|  U

## Q6: Count of records per city, only cities with count > 100

In [12]:
result = (df.groupBy("city")
            .count()
            .filter("count > 100"))
result.show()

+----+-----+
|city|count|
+----+-----+
+----+-----+



## Q7: Immutability and data cleaning (dropping/renaming columns)

DataFrames are immutable — you can't modify one in place. Every operation like dropping or renaming a column (`.drop()`, `.withColumnRenamed()`) returns a **new** DataFrame; the original is untouched. So cleaning steps are typically chained or reassigned. This also means Spark can safely build a lineage/DAG of transformations and optimize/reorder them, since nothing is mutated mid-pipeline.

In [13]:
df = df.drop("temp_col").withColumnRenamed("old_name", "new_name")

## Q8: Filter age between 18 and 30 (inclusive) and subscription == 'Premium'

In [14]:
result = df.filter((df.age >= 18) & (df.age <= 30) & (df.subscription == "Premium"))
result.show()

+-------+----------------+------+----------------+-----------+---+------------+-----------+-------------------+--------+------+--------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|age|subscription|       city|              email|username| price|store_id|  status|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---+------------+-----------+-------------------+--------+------+--------+--------+-------------------+
|  U1330|      10/11/2024|  East|        Clothing|     199.46| 24|     Premium| Georgetown|user330@example.com|user_330|689.66|    S014|Inactive|2024-10-20 04:00:00|
|  U1153|      01/03/2024| South|       Groceries|       55.6| 21|     Premium| Georgetown|user153@example.com|user_153|  36.0|    S012|  Active|2024-11-01 19:00:00|
|  U1271|      2024-08-13|  East|     Electronics|      144.0| 25|     Premium|   Franklin|user271@example.com|user_271|238.04|    S011|  Active|2024-09-03 21:00:00|
|  U

## Q9: Why handle nulls before mathematical aggregations (sum/avg)?

Functions like `sum()` and `avg()` either skip nulls silently (which can silently distort results if nulls are meaningful/missing-not-at-random) or propagate them depending on context. If you don't handle nulls explicitly first, you may get misleading averages/sums, inconsistent row counts across aggregates, or errors in downstream calculations. Cleaning nulls first (drop or fill with a sensible default) ensures aggregation results are accurate and reproducible.

## Q10: Cast raw_timestamp to TimestampType and rename to event_time

In [15]:
df = (df.withColumn("event_time", df["raw_timestamp"].cast(TimestampType()))
         .drop("raw_timestamp"))
df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



## Q11: The 'Shuffle' process in grouping operations — why a wide transformation?

When you perform a `groupBy` (or any operation requiring data with the same key to be co-located, like joins), Spark must redistribute (shuffle) data across partitions/nodes so that all records with the same key end up in the same partition for the reduce step. This involves writing intermediate data to disk and transferring it over the network between executors — expensive in I/O and network cost.

It's called a **wide transformation** because output partitions depend on data from *multiple* input partitions (unlike narrow transformations like `filter` or `map`, where each output partition depends on only one input partition).

## Q12: Remove rows where email is null OR username is an empty string

In [17]:
df_clean = df.filter(df.email.isNotNull() & (df.username != ""))
df_clean.show()

+-------+----------------+------+----------------+-----------+---+------------+-----------+-------------------+--------+------+--------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|age|subscription|       city|              email|username| price|store_id|  status|         event_time|
+-------+----------------+------+----------------+-----------+---+------------+-----------+-------------------+--------+------+--------+--------+-------------------+
|  U1330|      10/11/2024|  East|        Clothing|     199.46| 24|     Premium| Georgetown|user330@example.com|user_330|689.66|    S014|Inactive|2024-10-20 04:00:00|
|  U1369|      03/08/2024| North|       Groceries|       NULL| 23|        Free|   Franklin|user369@example.com|user_369|376.07|    S006| Pending|2025-02-04 22:00:00|
|  U1144|      2024-06-29|  East|           Books|       96.1| 50|        Free|Springfield|user144@example.com|user_144|819.82|    S002|  Active|2024-07-30 04:00:00|
|  U

## Q13: Use .agg() to calculate min, max, and mean of price in one call

In [18]:
df.agg(
     F.min("price").alias("min_price"),
     F.max("price").alias("max_price"),
     F.mean("price").alias("mean_price")
 ).show()

+---------+---------+-----------------+
|min_price|max_price|       mean_price|
+---------+---------+-----------------+
|    10.18|   999.25|490.4663581488935|
+---------+---------+-----------------+



## Q14: Risk of inferSchema=true with messy/inconsistent date formats

`inferSchema=true` samples the data to guess column types. If date formats are inconsistent (e.g., mixed `MM/DD/YYYY` and `YYYY-MM-DD`, or some rows with garbage/text), Spark may:

- Infer the column as `StringType` instead of `DateType`/`TimestampType`, silently skipping proper date parsing
- Throw parsing errors or produce nulls for rows that don't match the inferred format
- Give inconsistent behavior across runs if the sample used for inference differs

It's safer to explicitly define the schema (`StructType`) or parse dates manually with `to_date()` / `to_timestamp()` and a specified format.

## Q15: Final processing pipeline

1. Filter out duplicates
2. Fill null prices with 0
3. Group by `store_id` to calculate total revenue

In [19]:
result = (df
    .dropDuplicates()                            # 1. remove duplicates
    .na.fill({"price": 0})                        # 2. fill null prices with 0
    .groupBy("store_id")
    .agg(F.sum("price").alias("total_revenue"))   # 3. total revenue per store
)
result.show()

+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
|    S004|           15761.0|
|    S001|           9793.57|
|    S016|14011.480000000003|
|    S008|           9693.84|
|    S018|13368.730000000001|
|    S020|13255.260000000002|
|    S015|           9443.24|
|    S009|6757.1900000000005|
|    S002|          11237.05|
|    S005|          15719.18|
|    S006|           9829.07|
|    S003|17016.629999999997|
|    S014|14649.939999999999|
|    S017|          11306.29|
|    S013|           8388.31|
|    S011|           15797.6|
|    S007| 8697.469999999998|
|    S012| 7561.719999999999|
|    S019|12862.809999999998|
|    S010|11526.849999999997|
+--------+------------------+

